In [ ]:
# Cell 1: Load Encoded Dataset, Enforce Zero Leakage & Stratified 80/20 Split
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# 1. Load encoded dataset from Notebook 02 (NOT the clustered file)
df = pd.read_csv('data/processed/psu_financial_literacy_encoded.csv')

# 2. Strictly drop targets AND all q11-q42 items used to construct dfl_score
score_prefixes = tuple(f'q{i}_' for i in range(11, 43))
leakage_cols = [col for col in df.columns if col.startswith(score_prefixes)]
drop_cols = ['dfl_score', 'dfl_class'] + leakage_cols

X = df.drop(columns=drop_cols)
y = df['dfl_class']  # 0 = Below Target (<70), 1 = Meets Target (>=70)

# 3. Stratified 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Total respondents: {len(df)} | Leakage-free predictor features: {X.shape[1]}")
print(f"Dropped columns: {len(drop_cols)} (2 targets + {len(leakage_cols)} score-formula items q11-q42)")
print(f"Train set: {len(X_train)} rows | Below-70 count: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"Test set:  {len(X_test)} rows  | Below-70 count: {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)")

In [ ]:
# Cell 2: Define Logistic Regression & Random Forest Pipelines + Risk Metrics Helper
def build_models():
    smote = SMOTE(sampling_strategy=0.5, k_neighbors=3, random_state=42)
    return {
        'LR - Baseline (Raw)': ImbPipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=42))
        ]),
        'LR - Balanced Weights': ImbPipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=42))
        ]),
        'LR - In-Fold SMOTE': ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', smote),
            ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=42))
        ]),
        'RF - Baseline (Raw)': ImbPipeline([
            ('clf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=42
            ))
        ]),
        'RF - Balanced Weights': ImbPipeline([
            ('clf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=2,
                class_weight='balanced', random_state=42
            ))
        ]),
        'RF - In-Fold SMOTE': ImbPipeline([
            ('smote', smote),
            ('clf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=42
            ))
        ])
    }

def compute_risk_metrics(y_true, y_pred, y_prob_risk):
    """Evaluates detection of Class 0 (Below OECD 70/100 Target)."""
    y_true_risk = (y_true == 0).astype(int)
    return {
        'At-Risk Recall (C0)': round(recall_score(y_true, y_pred, pos_label=0, zero_division=0), 4),
        'At-Risk Precision (C0)': round(precision_score(y_true, y_pred, pos_label=0, zero_division=0), 4),
        'At-Risk F1 (C0)': round(f1_score(y_true, y_pred, pos_label=0, zero_division=0), 4),
        'Balanced Acc': round(balanced_accuracy_score(y_true, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_true_risk, y_prob_risk), 4),
        'At-Risk PR-AUC': round(average_precision_score(y_true_risk, y_prob_risk), 4)
    }

def find_best_threshold(y_true, prob_risk):
    """Finds the P(Class 0) cutoff maximizing Class 0 F1 strictly on training OOF probabilities."""
    best_t, best_f1 = 0.50, -1.0
    for t in np.arange(0.15, 0.56, 0.02):
        preds = np.where(prob_risk >= t, 0, 1)
        score = f1_score(y_true, preds, pos_label=0, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = t
    return round(float(best_t), 2), round(float(best_f1), 4)

print(f"Configured {len(build_models())} leakage-free classification pipelines.")

In [ ]:
# Cell 3: Stratified 5-Fold CV Benchmark (Inside 80% Training Split) & Threshold Selection
skf_train = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
oof_train_probs = {}
learned_thresholds = {}

for name, pipe in build_models().items():
    oof_preds = np.zeros(len(y_train), dtype=int)
    oof_prob_risk = np.zeros(len(y_train), dtype=float)

    for tr_idx, val_idx in skf_train.split(X_train, y_train):
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        oof_preds[val_idx] = pipe.predict(X_train.iloc[val_idx])
        oof_prob_risk[val_idx] = pipe.predict_proba(X_train.iloc[val_idx])[:, 0]

    oof_train_probs[name] = oof_prob_risk
    best_t, tuned_f1 = find_best_threshold(y_train.values, oof_prob_risk)
    learned_thresholds[name] = best_t

    metrics = compute_risk_metrics(y_train.values, oof_preds, oof_prob_risk)
    metrics['Tuned Cutoff'] = best_t
    metrics['Tuned Train F1'] = tuned_f1
    metrics['Model'] = name
    cv_results.append(metrics)

# Rank by threshold-independent ROC-AUC and PR-AUC so 0.50-cutoff ties never skew selection
cv_df = pd.DataFrame(cv_results).set_index('Model')
cv_df = cv_df.sort_values(by=['ROC-AUC', 'At-Risk PR-AUC'], ascending=False)

print("=== Stratified 5-Fold CV Results (80% Training Set, n = 94 | 11 At-Risk) ===")
print(cv_df.to_string())

In [ ]:
# Cell 4: Threshold-Tuned 20% Holdout Test & Full-Cohort 5-Fold OOF Evaluation
best_lr_name = cv_df.loc[cv_df.index.str.startswith('LR')].index[0]
best_rf_name = cv_df.loc[cv_df.index.str.startswith('RF')].index[0]
selected_models = [best_lr_name, best_rf_name]
print(f"Selected Champions from Training CV -> {best_lr_name}  |  {best_rf_name}\n")

# 1. Strict 20% Holdout Test Set (using frozen threshold learned strictly on 80% Training CV)
holdout_pipes = {}
tuned_holdout_rows = []

for name in selected_models:
    best_t = learned_thresholds[name]
    pipe = build_models()[name].fit(X_train, y_train)
    holdout_pipes[name] = pipe

    test_prob = pipe.predict_proba(X_test)[:, 0]
    test_preds = np.where(test_prob >= best_t, 0, 1)

    row = compute_risk_metrics(y_test.values, test_preds, test_prob)
    row['Threshold'] = best_t
    row['Model'] = name
    tuned_holdout_rows.append(row)

holdout_df = pd.DataFrame(tuned_holdout_rows).set_index('Model')
print("=== Threshold-Tuned 20% Holdout Test Set (n = 24 | 3 At-Risk) ===")
print(holdout_df.to_string())

# 2. Full-Cohort 5-Fold Out-of-Fold Evaluation & Tuned Confusion Matrices (n = 118 | All 14 At-Risk)
skf_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tuned_oof_rows = []

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for idx, name in enumerate(selected_models):
    best_t = learned_thresholds[name]
    full_preds = np.zeros(len(y), dtype=int)
    full_prob = np.zeros(len(y), dtype=float)

    for tr_idx, val_idx in skf_full.split(X, y):
        pipe_fold = build_models()[name].fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p_val = pipe_fold.predict_proba(X.iloc[val_idx])[:, 0]
        full_prob[val_idx] = p_val
        full_preds[val_idx] = np.where(p_val >= best_t, 0, 1)

    row = compute_risk_metrics(y.values, full_preds, full_prob)
    row['Threshold'] = best_t
    row['Model'] = name
    tuned_oof_rows.append(row)

    cm = confusion_matrix(y.values, full_preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Below 70 (0)', 'Meets 70 (1)'])
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f'{name}\n(Cutoff >= {best_t:.2f} | 5-Fold OOF, n = 118)')

tuned_oof_df = pd.DataFrame(tuned_oof_rows).set_index('Model')
print("\n=== Threshold-Tuned Full-Cohort 5-Fold OOF (n = 118 | All 14 At-Risk) ===")
print(tuned_oof_df.to_string())

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Top Predictive Drivers (Reusing Fitted Holdout Pipelines)
lr_pipe = holdout_pipes[best_lr_name]
rf_pipe = holdout_pipes[best_rf_name]

lr_coefs = pd.Series(lr_pipe.named_steps['clf'].coef_[0], index=X.columns)
rf_imp = pd.Series(rf_pipe.named_steps['clf'].feature_importances_, index=X.columns)

print(f"=== {best_lr_name}: Top 8 Predictors of Below-70 Risk (Negative Coef) ===")
print(lr_coefs.sort_values().head(8).round(4).to_string())

print(f"\n=== {best_lr_name}: Top 8 Predictors of Meeting 70/100 Target (Positive Coef) ===")
print(lr_coefs.sort_values(ascending=False).head(8).round(4).to_string())

print(f"\n=== {best_rf_name}: Top 12 Most Important Features ===")
print(rf_imp.sort_values(ascending=False).head(12).round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

top_lr_abs = lr_coefs.reindex(lr_coefs.abs().sort_values(ascending=False).head(10).index).sort_values()
colors = ['#c53030' if v < 0 else '#2b6cb0' for v in top_lr_abs.values]
axes[0].barh(top_lr_abs.index, top_lr_abs.values, color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title(f'{best_lr_name}\n(Red = Risk <70 | Blue = Pass >=70)')
axes[0].set_xlabel('Standardized Log-Odds Coefficient')

rf_imp.sort_values(ascending=False).head(10).sort_values().plot(kind='barh', ax=axes[1], color='#2f855a')
axes[1].set_title(f'{best_rf_name}\nTop 10 Feature Importances')
axes[1].set_xlabel('Mean Decrease in Impurity')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Export Threshold-Tuned Champion Pipeline & Chapter 4 Benchmark Tables
os.makedirs('models', exist_ok=True)

champion_name = tuned_oof_df.sort_values(
    by=['At-Risk F1 (C0)', 'ROC-AUC'],
    ascending=False
).index[0]
champion_threshold = float(learned_thresholds[champion_name])
champion_pipe = build_models()[champion_name].fit(X, y)

artifact = {
    'model_name': champion_name,
    'pipeline': champion_pipe,
    'feature_columns': list(X.columns),
    'decision_threshold': champion_threshold,
    'oecd_threshold': 70.0,
    'target_mapping': {0: 'Below Target (<70)', 1: 'Meets Target (>=70)'}
}

model_path = 'models/dfl_classifier_pipeline.joblib'
joblib.dump(artifact, model_path)

cv_df.to_csv('data/processed/ml_cv_benchmark.csv')
holdout_df.to_csv('data/processed/ml_tuned_holdout.csv')
tuned_oof_df.to_csv('data/processed/ml_tuned_oof_comparison.csv')

print(f"Exported champion model: {champion_name} (P(Risk) >= {champion_threshold:.2f})")
print(f"Saved artifact -> {model_path} ({len(artifact['feature_columns'])} input features)")
print("Exported Chapter 4 benchmark CSVs to data/processed/.")